# Hypothesis Testing, Chi-Square Test, and t-test

Dataset used: Pima Indians Diabetes dataset.

This notebook implements:
- Hypothesis testing framework
- Chi-square test of independence
- One-sample and independent two-sample t-tests

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
df = pd.read_csv("Assignment 3/pima_indians_diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 1) Data Preparation

We clean medically invalid zero values in selected columns by replacing them with median values.

In [2]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for c in cols:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [3]:
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print("\nOutcome counts:")
print(df["Outcome"].value_counts())

Rows: 768 Columns: 9

Outcome counts:
Outcome
0    500
1    268
Name: count, dtype: int64


## 2) Chi-Square Test of Independence

Question: Is diabetes outcome associated with glucose category?

- Null hypothesis (H0): Glucose category and outcome are independent.
- Alternate hypothesis (H1): Glucose category and outcome are associated.
- Significance level: 0.05

In [4]:
df["GlucoseCategory"] = pd.cut(
    df["Glucose"],
    bins=[0, 99, 125, 300],
    labels=["Normal", "Prediabetes", "Diabetes"]
)

contingency = pd.crosstab(df["GlucoseCategory"], df["Outcome"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Contingency Table:")
print(contingency)
print("\nChi-square statistic:", round(chi2, 4))
print("p-value:", round(p_value, 6))
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("Result: Reject H0 -> Significant association")
else:
    print("Result: Fail to reject H0 -> No significant association")

pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)

Contingency Table:
Outcome            0    1
GlucoseCategory          
Normal           178   14
Prediabetes      201   78
Diabetes         121  176

Chi-square statistic: 147.9083
p-value: 0.0
Degrees of freedom: 2
Result: Reject H0 -> Significant association


Outcome,0,1
GlucoseCategory,,
Normal,125.000000,67.000000
Prediabetes,181.640625,97.359375
Diabetes,193.359375,103.640625


## 3) One-Sample t-test

Question: Is average glucose significantly different from 110?

- Null hypothesis (H0): Mean glucose = 110
- Alternate hypothesis (H1): Mean glucose != 110
- Significance level: 0.05

In [5]:
t_stat_1, p_value_1 = stats.ttest_1samp(df["Glucose"], popmean=110)
print("Sample mean glucose:", round(df["Glucose"].mean(), 3))
print("t-statistic:", round(t_stat_1, 4))
print("p-value:", round(p_value_1, 6))

if p_value_1 < 0.05:
    print("Result: Reject H0 -> Mean glucose is significantly different from 110")
else:
    print("Result: Fail to reject H0 -> No significant difference from 110")

Sample mean glucose: 121.656
t-statistic: 10.6125
p-value: 0.0
Result: Reject H0 -> Mean glucose is significantly different from 110


## 4) Independent Two-Sample t-test

Question: Is mean BMI different between diabetic and non-diabetic groups?

- Null hypothesis (H0): Mean BMI (Outcome=1) = Mean BMI (Outcome=0)
- Alternate hypothesis (H1): Means are different
- Significance level: 0.05

In [6]:
bmi_diabetic = df[df["Outcome"] == 1]["BMI"]
bmi_non_diabetic = df[df["Outcome"] == 0]["BMI"]

t_stat_2, p_value_2 = stats.ttest_ind(bmi_diabetic, bmi_non_diabetic, equal_var=False)
print("Mean BMI (Outcome=1):", round(bmi_diabetic.mean(), 3))
print("Mean BMI (Outcome=0):", round(bmi_non_diabetic.mean(), 3))
print("t-statistic:", round(t_stat_2, 4))
print("p-value:", round(p_value_2, 6))

if p_value_2 < 0.05:
    print("Result: Reject H0 -> Mean BMI differs between groups")
else:
    print("Result: Fail to reject H0 -> No significant difference in mean BMI")

Mean BMI (Outcome=1): 35.384
Mean BMI (Outcome=0): 30.886
t-statistic: 9.0517
p-value: 0.0
Result: Reject H0 -> Mean BMI differs between groups


## 5) Final Conclusion

- The Chi-square test checks association between categorical variables.
- The one-sample t-test checks if a sample mean differs from a reference value.
- The independent t-test compares means of two independent groups.
- Decisions are made using p-value and significance level (0.05).

In [8]:
summary = pd.DataFrame({
    "Test": ["Chi-square", "One-sample t-test", "Independent t-test"],
    "p_value": [p_value, p_value_1, p_value_2]
})
summary["Decision_at_0.05"] = np.where(summary["p_value"] < 0.05, "Reject H0", "Fail to Reject H0")
summary

,Test,p_value,Decision_at_0.05
0,Chi-square,7.623052e-33,Reject H0
1,One-sample t-test,1.209730e-24,Reject H0
2,Independent t-test,2.552896e-18,Reject H0
